## ИИММ: практическая работа 7

Суть работы:  
Предстоит реализовать простые задачи детекции и верификации лиц, а также протестировать работу различных методов трекинга объектов в различных задачах.

План работы:  
1. Детекция + верификация.
2. MIEM Lookalike.
3. Анализ устойчивости распознавания лиц.  
a) Оценить точность работы методов из DeepFace на тестовом видео.  
b) Оценить точность работы методов из DeepFace на аугментированных данных.
4. Антиспуфинг.


In [2]:
import cv2
import os
import requests
import faiss
import pickle
import numpy as np
import pandas as pd
import streamlit as st

from deepface import DeepFace
from datetime import datetime
from PIL import Image

### Этап 1. Детекция + верификация

In [2]:
def download_yunet():
    model_url = "https://github.com/opencv/opencv_zoo/raw/refs/heads/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
    model_filename = "face_detection_yunet_2023mar.onnx"

    if not os.path.exists(model_filename):
        try:
            response = requests.get(model_url, stream=True)
            response.raise_for_status()
            with open(model_filename, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("Загрузка завершена успешно.")
        except Exception as e:
            print(f"Ошибка при загрузке: {e}")


download_yunet()

In [ ]:
REFERENCE_IMG_PATH = "my_face.jpg"
MODEL_PATH = "face_detection_yunet_2023mar.onnx"
MODEL_NAME = "Facenet512" # ArcFace, FaceNet, Facenet512
THRESHOLD = 0.5

In [4]:
detector = cv2.FaceDetectorYN.create(
    model=MODEL_PATH,
    config="",
    input_size=(320, 320),
    score_threshold=0.9,
    nms_threshold=0.3,
    top_k=50
)


cap = cv2.VideoCapture(0)

In [ ]:
print("Нажмите 'q' для выхода.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    height, width, _ = frame.shape
    detector.setInputSize((width, height))
    
    _, faces = detector.detect(frame)
    
    if faces is not None:
        for face in faces:

            coords = face[:4].astype(np.int32)
            x, y, w, h = coords
            
            x, y = max(0, x), max(0, y)
            face_img = frame[y:y+h, x:x+w]
            
            is_match = False
            try:

                result = DeepFace.verify(
                    img1_path=face_img, 
                    img2_path=REFERENCE_IMG_PATH, 
                    model_name=MODEL_NAME,
                    enforce_detection=True,
                    detector_backend='skip'
                )
                if result["verified"] and result["distance"] < THRESHOLD:
                    is_match = True
            except Exception as e:
                pass

            color = (0, 255, 0) if is_match else (0, 0, 255)
            label = "Me" if is_match else "Unknown"
            
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.imshow("Face Recognition", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Нажмите 'q' для выхода.


### Этап 2. MIEM lookalike

In [ ]:
!curl -L -o ./photo_staff.rar https://drive.usercontent.google.com/download?id=1gfu5WynF5ZMDBfcFiVWLuA2S_uVyVQZL&export=download

In [2]:
!tar -xf photo_staff.rar

In [ ]:
!curl -L -o ./photo_staff.csv https://drive.usercontent.google.com/download?id=1A6NivL1zPL6kLSLUOETaG_IWnJ-kg411&export=download

In [ ]:
def create_db(folder_path="photo_staff", csv_path="photo_staff.csv"):
   
    df = pd.read_csv(csv_path)
    
    file_to_name = dict(zip(df['filename'], df['name']))
    
    embeddings = []
    metadata = []

    print(f"Найдено записей в CSV: {len(df)}")
    
    for filename in os.listdir(folder_path):
        if filename in file_to_name:
            path = os.path.join(folder_path, filename)
            try:
                
                objs = DeepFace.represent(img_path=path, 
                                          model_name="ArcFace", 
                                          enforce_detection=True)
                
                embedding = objs[0]["embedding"]
                
                embeddings.append(embedding)
                
                metadata.append(file_to_name[filename])
                
                if len(metadata) % 10 == 0:
                    print(f"Обработано {len(metadata)} лиц...")
                    
            except Exception as e:
                print(f"Пропуск {filename}: лицо не обнаружено или ошибка {e}")

    if not embeddings:
        print("Эмбеддинги не созданы.")
        return

    embeddings = np.array(embeddings).astype('float32')
    faiss.normalize_L2(embeddings)
    dimension = embeddings.shape[1]
    
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    faiss.write_index(index, "employees.index")
    with open("metadata.pkl", "wb") as f:
        pickle.dump(metadata, f)
    
    print(f"Проиндексировано лиц: {len(metadata)}")

In [17]:
create_db()

Найдено записей в CSV: 464
Обработано 10 лиц...
Обработано 20 лиц...
Обработано 30 лиц...
Пропуск 132_miem.jpeg: лицо не обнаружено или ошибка Face could not be detected in photo_staff\132_miem.jpeg.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
Пропуск 133_miem.jpeg: лицо не обнаружено или ошибка Face could not be detected in photo_staff\133_miem.jpeg.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
Пропуск 134_zulip.jpeg: лицо не обнаружено или ошибка Face could not be detected in photo_staff\134_zulip.jpeg.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
Пропуск 136_miem.jpeg: лицо не обнаружено или ошибка Face could not be detected in photo_staff\136_miem.jpeg.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
Пропуск 137_miem.jpeg: лицо не обнаружено или ошибка Face could not 

In [19]:
%%writefile app.py

import streamlit as st
import cv2
import numpy as np
import faiss
import pickle
import pandas as pd
from deepface import DeepFace
from PIL import Image
import os

@st.cache_resource
def load_resources():
    index = faiss.read_index("employees.index")
    with open("metadata.pkl", "rb") as f:
        metadata = pickle.load(f)
    df = pd.read_csv("photo_staff.csv")
    return index, metadata, df

st.set_page_config(page_title="MIEM Face ID", layout="centered")
st.title("Система распознавания сотрудников МИЭМ")
st.divider()

index, metadata, df_info = load_resources()

uploaded_file = st.file_uploader("Загрузите фото для поиска...", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    img = Image.open(uploaded_file)
    st.image(img, caption='Ваше изображение', width=300)
    
    if st.button('🔍 Проверить по базе FAISS'):
        try:
            with st.spinner('Анализируем лицо...'):
                img_array = np.array(img.convert('RGB'))
                
                res = DeepFace.represent(img_array, model_name="ArcFace", enforce_detection=True)
                target_embedding = np.array([res[0]["embedding"]]).astype('float32')
                faiss.normalize_L2(target_embedding)
                
                distance, index_id = index.search(target_embedding, k=1)
                
                match_idx = index_id[0][0]
                best_match_name = metadata[match_idx]
                dist_score = distance[0][0]

                THRESHOLD = 1.2
                
                if dist_score < THRESHOLD:
                    st.success(f"Лицо распознано!")
                    
                    col1, col2 = st.columns(2)
                    with col1:
                        st.metric("Сотрудник", best_match_name)
                        st.write(f"Уровень сходства (L2): **{dist_score:.4f}**")
                    
                    with col2:
                        filename = df_info[df_info['name'] == best_match_name]['filename'].values[0]
                        photo_path = os.path.join("photo_staff", filename)
                        
                        if os.path.exists(photo_path):
                            st.image(photo_path, caption="Фото из личного дела")
                        else:
                            st.warning("Фото в локальной папке не найдено")
                else:
                    st.error("Совпадений не найдено (превышен порог расстояния)")
                    st.write(f"Ближайший результат: {best_match_name} (Dist: {dist_score:.4f})")

        except Exception as e:
            st.error(f"Ошибка: {e}")
            st.info("Убедитесь, что на фото отчетливо видно лицо.")

Overwriting app.py


### Этап 3а. Оценить точность работы методов DeepFace на тестовом видео

In [2]:
VIDEO_PATH = "olmikheeva.mp4"
CSV_PATH = "olmikheeva.csv"
PHOTOS_DB_PATH = "task3_photos"
MODEL_NAME = "ArcFace"

In [3]:
def create_db(db_path):
    embeddings = []
    names = []
    
    for f in os.listdir(db_path):
        if f.endswith(".pkl"): os.remove(os.path.join(db_path, f))

    for person_name in os.listdir(db_path):
        person_dir = os.path.join(db_path, person_name)
        if not os.path.isdir(person_dir): continue
        
        for img_name in os.listdir(person_dir):
            img_path = os.path.join(person_dir, img_name)
            try:
                res = DeepFace.represent(img_path=img_path, model_name=MODEL_NAME,
                                         detector_backend='retinaface',
                                         enforce_detection=True)
                embedding = np.array(res[0]["embedding"]).astype('float32')
                
                faiss.normalize_L2(embedding.reshape(1, -1))
                
                embeddings.append(embedding)
                names.append(person_name)
            except Exception as e:
                print(f"Пропуск {img_path}: лицо не найдено")

    embeddings = np.array(embeddings)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    faiss.write_index(index, "vector_db.index")
    with open("metadata.pkl", "wb") as f:
        pickle.dump(names, f)
    print(f"Индексировано лиц: {len(names)}")

In [4]:
create_db(PHOTOS_DB_PATH)

Пропуск task3_photos\palmasizade\.ipynb_checkpoints: лицо не найдено
Индексировано лиц: 29


In [6]:
def time_to_seconds(t_str):
    t = datetime.strptime(t_str, "%H:%M:%S")
    return t.hour * 3600 + t.minute * 60 + t.second

def clean_persons(p_str):
    if pd.isna(p_str): return []
    return [p.strip().replace('"', '') for p in p_str.split(',')]

In [7]:
df_labels = pd.read_csv(CSV_PATH)
df_labels['from_sec'] = df_labels['from'].apply(time_to_seconds)
df_labels['to_sec'] = df_labels['to'].apply(time_to_seconds)

df_labels['persons_list'] = df_labels['persons'].apply(clean_persons)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

tp, fn = 0, 0
frame_idx = 0

index = faiss.read_index("vector_db.index")
with open("metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

In [8]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    if frame_idx % 50 == 0:
        curr_sec = frame_idx / fps
        row = df_labels[(df_labels['from_sec'] <= curr_sec) & (df_labels['to_sec'] >= curr_sec)]
        
        if not row.empty:
            expected_persons = row.iloc[0]['persons_list']
            found_names_on_frame = set()

            try:
                faces = DeepFace.represent(img_path=frame, model_name="ArcFace", 
                                           enforce_detection=False, detector_backend='opencv')
                
                for face in faces:
                    target_emb = np.array(face["embedding"]).astype('float32').reshape(1, -1)
                    faiss.normalize_L2(target_emb)
                    
                    distances, indices = index.search(target_emb, k=1)
                    dist = distances[0][0]
                    match_name = metadata[indices[0][0]]
                    
                    if dist < 0.95: 
                        found_names_on_frame.add(match_name)

                for p in expected_persons:
                    if p in found_names_on_frame:
                        tp += 1
                    else:
                        fn += 1
                        print(f"MISS: Sec {curr_sec:.1f} | Ожидали {p} | Нашли {found_names_on_frame}")

            except Exception as e:
                fn += len(expected_persons)

cap.release()

MISS: Sec 0.0 | Ожидали olmikheeva | Нашли set()
MISS: Sec 0.0 | Ожидали saaksenov | Нашли set()
MISS: Sec 0.0 | Ожидали olmikheeva | Нашли set()
MISS: Sec 0.0 | Ожидали saaksenov | Нашли set()
MISS: Sec 0.0 | Ожидали olmikheeva | Нашли set()
MISS: Sec 0.0 | Ожидали saaksenov | Нашли set()
MISS: Sec 0.0 | Ожидали olmikheeva | Нашли set()
MISS: Sec 0.0 | Ожидали saaksenov | Нашли set()
MISS: Sec 0.0 | Ожидали olmikheeva | Нашли set()
MISS: Sec 0.0 | Ожидали saaksenov | Нашли set()
MISS: Sec 0.0 | Ожидали olmikheeva | Нашли set()
MISS: Sec 0.0 | Ожидали saaksenov | Нашли set()
MISS: Sec 0.0 | Ожидали olmikheeva | Нашли set()
MISS: Sec 0.0 | Ожидали saaksenov | Нашли set()
MISS: Sec 0.0 | Ожидали olmikheeva | Нашли set()
MISS: Sec 0.0 | Ожидали saaksenov | Нашли set()
MISS: Sec 0.0 | Ожидали olmikheeva | Нашли set()
MISS: Sec 0.0 | Ожидали saaksenov | Нашли set()


KeyboardInterrupt: 

In [6]:
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"Результаты для {VIDEO_PATH}:")
print(f"True Positives: {tp}")
print(f"False Negatives: {fn}")
print(f"Метрика RECALL: {recall:.4f}")

Результаты для olmikheeva.mp4:
True Positives: 0
False Negatives: 705
Метрика RECALL: 0.0000


### Этап 3б. Оценить точность работы методов из DeepFace на аугментированных данных

In [3]:
def apply_augmentation(image, mode):
    if mode == "rotate_45":
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 45, 1.0)
        return cv2.warpAffine(image, M, (w, h))
    
    elif mode == "rotate_90":
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    
    elif mode == "noise":
        gauss = np.random.normal(0, 25, image.shape).astype('uint8')
        return cv2.add(image, gauss)
    
    elif mode == "brightness_plus":
        return cv2.convertScaleAbs(image, alpha=1, beta=50)
    
    elif mode == "brightness_minus":
        return cv2.convertScaleAbs(image, alpha=1, beta=-50)
    
    elif mode == "blur":
        return cv2.GaussianBlur(image, (15, 15), 0)
    
    return image

In [4]:
models = ["VGG-Face", "Facenet", "Facenet512", "OpenFace", "DeepFace", "DeepID", "ArcFace", "Dlib", "SFace", "GhostFaceNet"]

augmentations = {
    "Исходный датасет": "original",
    "Поворот на 45°": "rotate_45",
    "Поворот на 90°": "rotate_90",
    "Изображение с шумом": "noise",
    "Яркость +50%": "brightness_plus",
    "Яркость -50%": "brightness_minus",
    "Размытие": "blur"
}

REFERENCE_PATH = "passport_photo.jpg"
DATASET_DIR = "my_faces_20"

In [ ]:
results_table = pd.DataFrame(index=models, columns=augmentations.keys())

for model in models:
    print(f"Тестирование модели: {model}")
    for aug_name, aug_mode in augmentations.items():
        tp = 0
        total = 0
        
        for img_name in os.listdir(DATASET_DIR):
            img_path = os.path.join(DATASET_DIR, img_name)
            img = cv2.imread(img_path)
            if img is None: continue

            processed_img = apply_augmentation(img, aug_mode)
            
            try:
                res = DeepFace.verify(img1_path=REFERENCE_PATH, 
                                      img2_path=processed_img, 
                                      model_name=model, 
                                      enforce_detection=False,
                                      detector_backend='retinaface')
                if res["verified"]:
                    tp += 1
            except Exception as e:
                print(f"❌ Ошибка: {model} | {aug_name} | {img_name}: {e}")
            total += 1

        score = tp / total if total > 0 else 0
        results_table.loc[model, aug_name] = round(score, 2)

Тестирование модели: VGG-Face


NameError: name 'e' is not defined

In [6]:
print(results_table)

             Исходный датасет Поворот на 45° Поворот на 90°  \
VGG-Face                  0.0            0.0            0.0   
Facenet                   0.0            0.0            0.0   
Facenet512                0.0            0.0            0.0   
OpenFace                  0.0            0.0            0.0   
DeepFace                  0.0            0.0            0.0   
DeepID                    0.0            0.0            0.0   
ArcFace                   0.0            0.0            0.0   
Dlib                      0.0            0.0            0.0   
SFace                     0.0            0.0            0.0   
GhostFaceNet              0.0            0.0            0.0   

             Изображение с шумом Яркость +50% Яркость -50% Размытие  
VGG-Face                     0.0          0.0          0.0      0.0  
Facenet                      0.0          0.0          0.0      0.0  
Facenet512                   0.0          0.0          0.0      0.0  
OpenFace                  

In [ ]:
results_table.to_csv("face_metrics_results.csv")